# 03 — OECD Macro KPI Extraction

This notebook extracts macroeconomic indicators from OECD APIs to support the car registrations forecasting model.

The extracted indicators include:

- Consumer price index / HICP
- Short-term interest rates
- GDP growth
- Real final consumption growth
- Unemployment rates
- Consumer confidance
- Fuel prices (From EC Bulletin)

The outputs are saved as structured datasets and later merged with ACEA registration data.

In [1]:
import requests
import pandas as pd
from io import StringIO
from pathlib import Path

# Helper Functions For All OCED Extractions

In [2]:
def fetch_oecd_csv(url: str, cache_file: Path | None = None, force_refresh: bool = False) -> pd.DataFrame:
    if cache_file and cache_file.exists() and not force_refresh:
        print(f"Loading cached file: {cache_file}")
        return pd.read_csv(cache_file)

    print("Requesting OECD API...")
    response = requests.get(url, timeout=60)
    response.raise_for_status()

    df = pd.read_csv(StringIO(response.text))

    if cache_file:
        df.to_csv(cache_file, index=False)
        print(f"Saved cache to: {cache_file}")

    return df

## OECD Unemployment Rates Extraction

In [3]:
# ----------------------------
# CONFIG
# ----------------------------
START_PERIOD = "2022-01"
END_PERIOD = "2025-12"

# Your OECD unemployment selection copied from the Data Explorer
AGENCY = "OECD.SDD.TPS"
DATASET = "DSD_LFS@DF_IALFS_UNE_M"
VERSION = "1.0"
SELECTION = ".UNE_LF_M.._Z.Y._T.Y_GE15..M"

BASE_URL = "https://sdmx.oecd.org/public/rest/data"


In [4]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver" / "macro_kpis"

CACHE_DIR = BRONZE_DIR / "oecd_cache"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
SILVER_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
# CSV with both codes + labels is easiest for inspection
DATA_URL = (
    f"{BASE_URL}/{AGENCY},{DATASET},{VERSION}/{SELECTION}"
    f"?startPeriod={START_PERIOD}"
    f"&endPeriod={END_PERIOD}"
    f"&dimensionAtObservation=AllDimensions"
    f"&format=csvfilewithlabels"
)

CACHE_FILE = CACHE_DIR / f"unemployment_{START_PERIOD}_{END_PERIOD}.csv"




def make_unique_columns(columns):
    seen = {}
    new_cols = []

    for col in columns:
        if col not in seen:
            seen[col] = 0
            new_cols.append(col)
        else:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")
    return new_cols

def standardize_oecd_unemployment(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize OECD unemployment dataset into a cleaner structure.
    Handles duplicate column names from csvfilewithlabels.
    """
    df = df.copy()

    # Normalize + deduplicate column names
    normalized_cols = [c.strip().lower().replace(" ", "_") for c in df.columns]
    df.columns = make_unique_columns(normalized_cols)

    print("\nColumns returned by OECD API:")
    print(df.columns.tolist())


    expected_candidates = {
        "country_code": ["ref_area", "country", "location"],
        "country_label": ["reference_area", "ref_area_label", "country_label"],
        "time_period": ["time_period", "period", "time"],
        "value": ["obs_value", "value", "obsvalue"],
        "measure_code": ["measure"],
        "measure_label": ["measure_1", "measure_label"],
        "frequency": ["freq"],
        "frequency_label": ["frequency_of_observation"],
        "sex_code": ["sex"],
        "sex_label": ["sex_1"],
        "age_code": ["age"],
        "age_label": ["age_1"],
        "unit_code": ["unit_measure"],
        "unit_label": ["unit_of_measure"],
    }

    resolved = {}
    for target, options in expected_candidates.items():
        resolved[target] = next((c for c in options if c in df.columns), None)

    out = pd.DataFrame()

    if resolved["country_code"]:
        out["country_code"] = df[resolved["country_code"]]
    if resolved["country_label"]:
        out["country"] = df[resolved["country_label"]]
    if resolved["time_period"]:
        out["time_period"] = df[resolved["time_period"]]
    if resolved["value"]:
        out["value"] = pd.to_numeric(df[resolved["value"]], errors="coerce")
    if resolved["frequency"]:
        out["frequency_code"] = df[resolved["frequency"]]
    if resolved["frequency_label"]:
        out["frequency"] = df[resolved["frequency_label"]]
    if resolved["measure_code"]:
        out["measure_code"] = df[resolved["measure_code"]]
    if resolved["measure_label"]:
        out["measure"] = df[resolved["measure_label"]]
    if resolved["sex_code"]:
        out["sex_code"] = df[resolved["sex_code"]]
    if resolved["sex_label"]:
        out["sex"] = df[resolved["sex_label"]]
    if resolved["age_code"]:
        out["age_code"] = df[resolved["age_code"]]
    if resolved["age_label"]:
        out["age"] = df[resolved["age_label"]]
    if resolved["unit_code"]:
        out["unit_code"] = df[resolved["unit_code"]]
    if resolved["unit_label"]:
        out["unit"] = df[resolved["unit_label"]]

    # Add date helpers
    if "time_period" in out.columns:
        out["date"] = pd.to_datetime(out["time_period"], errors="coerce")
        out["year"] = out["date"].dt.year
        out["month"] = out["date"].dt.month
        out["quarter"] = out["date"].dt.quarter

    return out


In [6]:

# ----------------------------
# MAIN
# ----------------------------
if __name__ == "__main__":
    print("Testing OECD unemployment API...\n")
    print(DATA_URL)
    print()

    raw_df = fetch_oecd_csv(DATA_URL, cache_file=CACHE_FILE, force_refresh=False)
    print("\nRaw shape:", raw_df.shape)
    print(raw_df.head())

    Unemployment_df = standardize_oecd_unemployment(raw_df)
    print("\nClean shape:", Unemployment_df.shape)
    print(Unemployment_df.head())

    # Optional: save cleaned version
    output_file = CACHE_DIR / f"unemployment_clean_{START_PERIOD}_{END_PERIOD}.csv"
    Unemployment_df.to_csv(output_file, index=False)
    print(f"\nSaved cleaned data to: {output_file}")

Testing OECD unemployment API...

https://sdmx.oecd.org/public/rest/data/OECD.SDD.TPS,DSD_LFS@DF_IALFS_UNE_M,1.0/.UNE_LF_M.._Z.Y._T.Y_GE15..M?startPeriod=2022-01&endPeriod=2025-12&dimensionAtObservation=AllDimensions&format=csvfilewithlabels

Requesting OECD API...
Saved cache to: c:\Users\Bruger\OneDrive\Documents\Projects\Car Registrations Predictive Model\acea_github_repo_draft\notebooks\data\bronze\oecd_cache\unemployment_2022-01_2025-12.csv

Raw shape: (2061, 34)
  STRUCTURE                              STRUCTURE_ID  \
0  DATAFLOW  OECD.SDD.TPS:DSD_LFS@DF_IALFS_UNE_M(1.0)   
1  DATAFLOW  OECD.SDD.TPS:DSD_LFS@DF_IALFS_UNE_M(1.0)   
2  DATAFLOW  OECD.SDD.TPS:DSD_LFS@DF_IALFS_UNE_M(1.0)   
3  DATAFLOW  OECD.SDD.TPS:DSD_LFS@DF_IALFS_UNE_M(1.0)   
4  DATAFLOW  OECD.SDD.TPS:DSD_LFS@DF_IALFS_UNE_M(1.0)   

               STRUCTURE_NAME ACTION REF_AREA Reference area   MEASURE  \
0  Monthly unemployment rates      I      JPN          Japan  UNE_LF_M   
1  Monthly unemployment rates      I

In [ ]:
display(
    Unemployment_df[Unemployment_df["country"].str.contains("European Union", na=False)]
    .sort_values(["year", "month"])[["country", "year", "month", "quarter", "value", "age"]]
.head(20))

## OECD Consumer Confidence Extraction

In [ ]:

# ----------------------------
# CONFIG
# ----------------------------
START_PERIOD = "2022-01"
END_PERIOD = "2025-12"

AGENCY = "OECD.SDD.STES"
DATASET = "DSD_STES@DF_CS"
VERSION = "4.0"
SELECTION = ".M.CCICP......"

BASE_URL = "https://sdmx.oecd.org/public/rest/data"
CACHE_DIR = Path("oecd_cache")
CACHE_DIR.mkdir(exist_ok=True)

DATA_URL = (
    f"{BASE_URL}/{AGENCY},{DATASET},{VERSION}/{SELECTION}"
    f"?startPeriod={START_PERIOD}"
    f"&endPeriod={END_PERIOD}"
    f"&dimensionAtObservation=AllDimensions"
    f"&format=csvfilewithlabels"
)

CACHE_FILE = CACHE_DIR / f"consumer_confidence_{START_PERIOD}_{END_PERIOD}.csv"




def make_unique_columns(columns):
    seen = {}
    new_cols = []

    for col in columns:
        if col not in seen:
            seen[col] = 0
            new_cols.append(col)
        else:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")
    return new_cols


def standardize_oecd_consumer_confidence(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    normalized_cols = [c.strip().lower().replace(" ", "_") for c in df.columns]
    df.columns = make_unique_columns(normalized_cols)

    print("\nColumns returned by OECD API:")
    print(df.columns.tolist())

    # Adjust these if the returned columns differ slightly
    out = pd.DataFrame({
        "country_code": df["ref_area"],
        "country": df["reference_area"],
        "time_period": df["time_period"],
        "value": pd.to_numeric(df["obs_value"], errors="coerce"),
    })

    if "unit_of_measure" in df.columns:
        out["unit"] = df["unit_of_measure"]
    elif "unit_measure_1" in df.columns:
        out["unit"] = df["unit_measure_1"]

    if "measure_1" in df.columns:
        out["measure"] = df["measure_1"]
    elif "measure" in df.columns:
        out["measure"] = df["measure"]

    if "frequency_of_observation" in df.columns:
        out["frequency"] = df["frequency_of_observation"]
    elif "freq" in df.columns:
        out["frequency"] = df["freq"]

    out["date"] = pd.to_datetime(out["time_period"], errors="coerce")
    out["year"] = out["date"].dt.year
    out["month"] = out["date"].dt.month
    out["quarter"] = out["date"].dt.quarter

    out["indicator_name"] = "Composite consumer confidence"
    out["dataset_id"] = DATASET
    out["source"] = "OECD API"

    return out


print("Fetching OECD Consumer Confidance data...\n")
print(DATA_URL)

raw_df = fetch_oecd_csv(DATA_URL, cache_file=CACHE_FILE, force_refresh=False)

print("\nRaw shape:", raw_df.shape)

CCI_df = standardize_oecd_unemployment(raw_df)

print("\nClean shape:", CCI_df.shape)
display(CCI_df.head())

output_file = CACHE_DIR / f"consumer_confidence_clean_{START_PERIOD}_{END_PERIOD}.csv"
CCI_df.to_csv(output_file, index=False)

print(f"\nSaved cleaned data to: {output_file}")

In [ ]:
display(CCI_df.head(10))

# ## OECD Consumer Price Index (Inflation) API Extraction

In [ ]:

# ----------------------------
# CONFIG
# ----------------------------
START_PERIOD = "2022-01"
END_PERIOD = "2025-12"

AGENCY = "OECD.SDD.TPS"
DATASET = "DSD_PRICES_COICOP2018@DF_PRICES_C2018_ALL"
VERSION = "1.0"
SELECTION = ".M.HICP.CPI.PA..."

BASE_URL = "https://sdmx.oecd.org/public/rest/data"
CACHE_DIR = Path("oecd_cache")
CACHE_DIR.mkdir(exist_ok=True)

DATA_URL = (
    f"https://sdmx.oecd.org/public/rest/data/"
    f"{AGENCY},{DATASET},{VERSION}/{SELECTION}"
    f"?startPeriod=2022-01"
    f"&endPeriod=2025-12"
    f"&dimensionAtObservation=AllDimensions"
    f"&format=csvfilewithlabels"
)

CACHE_FILE = CACHE_DIR / f"consumer_Price_Index_{START_PERIOD}_{END_PERIOD}.csv"



def make_unique_columns(columns):
    seen = {}
    new_cols = []

    for col in columns:
        if col not in seen:
            seen[col] = 0
            new_cols.append(col)
        else:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")
    return new_cols


def standardize_oecd_consumer_price_index(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    normalized_cols = [c.strip().lower().replace(" ", "_") for c in df.columns]
    df.columns = make_unique_columns(normalized_cols)

    # ----------------------------
    # FILTER TO THE CORRECT CPI SERIES
    # ----------------------------

    # Methodology = HICP
    if "methodology" in df.columns:
        df = df[df["methodology"] == "HICP"].copy()
    elif "methodology_1" in df.columns:
        df = df[df["methodology_1"] == "HICP"].copy()

    # Expenditure = Total
    if "expenditure" in df.columns:
        df = df[df["expenditure"].str.contains("_T", case=False, na=False)].copy()
    elif "expenditure_1" in df.columns:
        df = df[df["expenditure_1"].str.contains("_T", case=False, na=False)].copy()

    # Transformation = Growth rate over 1 year
    if "transformation_1" in df.columns:
        df = df[df["transformation_1"].str.contains("over 1 year", case=False, na=False)].copy()
    elif "transformation" in df.columns:
        df = df[df["transformation"].str.contains("over 1 year", case=False, na=False)].copy()

    # Unit = Percent per annum
    if "unit_of_measure" in df.columns:
        df = df[df["unit_of_measure"] == "Percent per annum"].copy()
    elif "unit_measure_1" in df.columns:
        df = df[df["unit_measure_1"] == "Percent per annum"].copy()

    # ----------------------------
    # BUILD OUTPUT
    # ----------------------------

    # Adjust these if the returned columns differ slightly
    out = pd.DataFrame({
        "country_code": df["ref_area"],
        "country": df["reference_area"],
        "time_period": df["time_period"],
        "value": pd.to_numeric(df["obs_value"], errors="coerce"),
    })

    if "unit_of_measure" in df.columns:
        out["unit"] = df["unit_of_measure"]
    elif "unit_measure_1" in df.columns:
        out["unit"] = df["unit_measure_1"]

    if "measure_1" in df.columns:
        out["measure"] = df["measure_1"]
    elif "measure" in df.columns:
        out["measure"] = df["measure"]

    if "frequency_of_observation" in df.columns:
        out["frequency"] = df["frequency_of_observation"]
    elif "freq" in df.columns:
        out["frequency"] = df["freq"]

    out["date"] = pd.to_datetime(out["time_period"], errors="coerce")
    out["year"] = out["date"].dt.year
    out["month"] = out["date"].dt.month
    out["quarter"] = out["date"].dt.quarter

    out["indicator_name"] = "consumer_Price_Index"
    out["dataset_id"] = DATASET
    out["source"] = "OECD API"

    return out


if __name__ == "__main__":
    print("Testing OECD consumer price index API...\n")
    print(DATA_URL)
    print()

    raw_df = fetch_oecd_csv(DATA_URL, cache_file=CACHE_FILE, force_refresh=False)
    print("\nRaw shape:", raw_df.shape)
    print(raw_df.head())

    CPI_df = standardize_oecd_consumer_price_index(raw_df)
    print("\nClean shape:", CPI_df.shape)
    print(CPI_df.head())

    output_file = CACHE_DIR / f"consumer_Price_Index_clean_{START_PERIOD}_{END_PERIOD}.csv"
    CPI_df.to_csv(output_file, index=False)
    print(f"\nSaved cleaned data to: {output_file}")

In [ ]:
display(
    CPI_df[
        (CPI_df["country"].str.contains("European Union", na=False)) &
        (CPI_df["year"] == 2022)
    ][["country", "month", "year", "unit", "measure", "value"]]
   .tail(12)
)

# OCED Interest Rates API Extraction

In [ ]:

# ----------------------------
# CONFIG
# ----------------------------
START_PERIOD = "2022-01"
END_PERIOD = "2025-12"

AGENCY = "OECD.SDD.STES"
DATASET = "DSD_STES@DF_FINMARK"
VERSION = "4.0"
SELECTION = ".M.IR3TIB.PA....."

BASE_URL = "https://sdmx.oecd.org/public/rest/data"
CACHE_DIR = Path("oecd_cache")
CACHE_DIR.mkdir(exist_ok=True)

DATA_URL = (
    f"https://sdmx.oecd.org/public/rest/data/"
    f"{AGENCY},{DATASET},{VERSION}/{SELECTION}"
    f"?startPeriod=2022-01"
    f"&endPeriod=2025-12"
    f"&dimensionAtObservation=AllDimensions"
    f"&format=csvfilewithlabels"
)

CACHE_FILE = CACHE_DIR / f"interest_rates_Index_{START_PERIOD}_{END_PERIOD}.csv"




def make_unique_columns(columns):
    seen = {}
    new_cols = []

    for col in columns:
        if col not in seen:
            seen[col] = 0
            new_cols.append(col)
        else:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")
    return new_cols


def standardize_oecd_consumer_Price_Index(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    normalized_cols = [c.strip().lower().replace(" ", "_") for c in df.columns]
    df.columns = make_unique_columns(normalized_cols)

    print("\nColumns returned by OECD API:")
    print(df.columns.tolist())

    # Adjust these if the returned columns differ slightly
    out = pd.DataFrame({
        "country_code": df["ref_area"],
        "country": df["reference_area"],
        "time_period": df["time_period"],
        "value": pd.to_numeric(df["obs_value"], errors="coerce"),
    })

    if "unit_of_measure" in df.columns:
        out["unit"] = df["unit_of_measure"]
    elif "unit_measure_1" in df.columns:
        out["unit"] = df["unit_measure_1"]

    if "measure_1" in df.columns:
        out["measure"] = df["measure_1"]
    elif "measure" in df.columns:
        out["measure"] = df["measure"]

    if "frequency_of_observation" in df.columns:
        out["frequency"] = df["frequency_of_observation"]
    elif "freq" in df.columns:
        out["frequency"] = df["freq"]

    out["date"] = pd.to_datetime(out["time_period"], errors="coerce")
    out["year"] = out["date"].dt.year
    out["month"] = out["date"].dt.month
    out["quarter"] = out["date"].dt.quarter

    out["indicator_name"] = "interest_rates_Index"
    out["dataset_id"] = DATASET
    out["source"] = "OECD API"

    return out


if __name__ == "__main__":
    print("Testing OECD interest rates index API...\n")
    print(DATA_URL)
    print()

    ir_raw_df = fetch_oecd_csv(DATA_URL, cache_file=CACHE_FILE, force_refresh=False)
    print("\nRaw shape:", ir_raw_df.shape)
    print(ir_raw_df.head())

    interest_rates_df = standardize_oecd_consumer_Price_Index(ir_raw_df )
    print("\nClean shape:", interest_rates_df.shape)
    print(interest_rates_df.head())

    output_file = CACHE_DIR / f"Interest_Rates_Index_clean_{START_PERIOD}_{END_PERIOD}.csv"
    interest_rates_df.to_csv(output_file, index=False)
    print(f"\nSaved cleaned data to: {output_file}")

In [ ]:
display(interest_rates_df)

# OCED GDP API Extraction

In [ ]:

# ----------------------------
# CONFIG
# ----------------------------
START_PERIOD = "2022-Q1"
END_PERIOD = "2025-Q4"

AGENCY = "OECD.SDD.NAD"
DATASET = "DSD_NAMAIN1@DF_QNA_EXPENDITURE_GROWTH_OECD"
VERSION = "1.1"
SELECTION = "Q..AUS+AUT+BEL+CAN+CHE+CHL+COL+CRI+CZE+DEU+DNK+ESP+FIN+EST+FRA+GBR+GRC+HUN+ISL+LTU+IRL+ISR+ITA+JPN+KOR+LUX+LVA+MEX+NLD+NOR+NZL+POL+PRT+SVK+SVN+SWE+TUR+USA+OECD+G20+G7+USMCA+OECDE+EA20+EU27_2020.S1..B1GQ......GY."

BASE_URL = "https://sdmx.oecd.org/public/rest/data"
CACHE_DIR = Path("oecd_cache")
CACHE_DIR.mkdir(exist_ok=True)

DATA_URL = (
    f"https://sdmx.oecd.org/public/rest/data/"
    f"{AGENCY},{DATASET},{VERSION}/{SELECTION}"
    f"?startPeriod=2022-01"
    f"&endPeriod=2025-12"
    f"&dimensionAtObservation=AllDimensions"
    f"&format=csvfilewithlabels"
)

CACHE_FILE = CACHE_DIR / f"GDP_Index_{START_PERIOD}_{END_PERIOD}.csv"



def make_unique_columns(columns):
    seen = {}
    new_cols = []

    for col in columns:
        if col not in seen:
            seen[col] = 0
            new_cols.append(col)
        else:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")
    return new_cols


def standardize_oecd_consumer_Price_Index(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    normalized_cols = [c.strip().lower().replace(" ", "_") for c in df.columns]
    df.columns = make_unique_columns(normalized_cols)

    print("\nColumns returned by OECD API:")
    print(df.columns.tolist())

    # Adjust these if the returned columns differ slightly
    out = pd.DataFrame({
        "country_code": df["ref_area"],
        "country": df["reference_area"],
        "time_period": df["time_period"],
        "value": pd.to_numeric(df["obs_value"], errors="coerce"),
    })

    if "unit_of_measure" in df.columns:
        out["unit"] = df["unit_of_measure"]
    elif "unit_measure_1" in df.columns:
        out["unit"] = df["unit_measure_1"]

    if "measure_1" in df.columns:
        out["measure"] = df["measure_1"]
    elif "measure" in df.columns:
        out["measure"] = df["measure"]

    if "frequency_of_observation" in df.columns:
        out["frequency"] = df["frequency_of_observation"]
    elif "freq" in df.columns:
        out["frequency"] = df["freq"]

    out["date"] = pd.to_datetime(out["time_period"], errors="coerce")
    out["year"] = out["date"].dt.year
    out["month"] = out["date"].dt.month
    out["quarter"] = out["date"].dt.quarter

    out["indicator_name"] = "GDP_Index"
    out["dataset_id"] = DATASET
    out["source"] = "OECD API"

    return out


if __name__ == "__main__":
    print("Testing OECD GDP index API...\n")
    print(DATA_URL)
    print()

    gdp_raw_df = fetch_oecd_csv(DATA_URL, cache_file=CACHE_FILE, force_refresh=False)
    print("\nRaw shape:", gdp_raw_df.shape)
    print(gdp_raw_df.head())

    gdp_df = standardize_oecd_consumer_Price_Index(gdp_raw_df )
    print("\nClean shape:", gdp_df.shape)
    print(gdp_df.head())

    output_file = CACHE_DIR / f"GDP_Index_clean_{START_PERIOD}_{END_PERIOD}.csv"
    gdp_df.to_csv(output_file, index=False)
    print(f"\nSaved cleaned data to: {output_file}")

In [ ]:
gdp_df = gdp_df.sort_values(["country", "date"])

gdp_df = (
    gdp_df.set_index("date")
          .groupby("country", group_keys=False)
          .resample("ME")
          .ffill()
          .reset_index()
)

In [ ]:
gdp_monthly_df = gdp_df.copy()

gdp_monthly_df["year"] = gdp_df["date"].dt.year
gdp_monthly_df["month"] = gdp_df["date"].dt.month
gdp_monthly_df["quarter"] = gdp_df["date"].dt.quarter

In [ ]:
display(gdp_monthly_df.head(5))

# OCED Real final consumption expenditure per capita of households API Extraction

In [ ]:

# ----------------------------
# CONFIG
# ----------------------------
START_PERIOD = "2022-Q1"
END_PERIOD = "2025-Q4"

AGENCY = "OECD.SDD.NAD"
DATASET = "DSD_HHDASH@DF_HHDASH_INDIC"
VERSION = "1.0"
SELECTION = "Q..P3S1M_R_POP_GR.PC"

BASE_URL = "https://sdmx.oecd.org/public/rest/data"
CACHE_DIR = Path("oecd_cache")
CACHE_DIR.mkdir(exist_ok=True)

DATA_URL = (
    f"https://sdmx.oecd.org/public/rest/data/"
    f"{AGENCY},{DATASET},{VERSION}/{SELECTION}"
    f"?startPeriod=2022-Q1"
    f"&endPeriod=2025-Q4"
    f"&dimensionAtObservation=AllDimensions"
    f"&format=csvfilewithlabels"
)

CACHE_FILE = CACHE_DIR / f"RFC_Index_{START_PERIOD}_{END_PERIOD}.csv"



def make_unique_columns(columns):
    seen = {}
    new_cols = []

    for col in columns:
        if col not in seen:
            seen[col] = 0
            new_cols.append(col)
        else:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")
    return new_cols


def standardize_oecd_consumer_Price_Index(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    normalized_cols = [c.strip().lower().replace(" ", "_") for c in df.columns]
    df.columns = make_unique_columns(normalized_cols)

    print("\nColumns returned by OECD API:")
    print(df.columns.tolist())

    # Adjust these if the returned columns differ slightly
    out = pd.DataFrame({
        "country_code": df["ref_area"],
        "country": df["reference_area"],
        "time_period": df["time_period"],
        "value": pd.to_numeric(df["obs_value"], errors="coerce"),
    })

    if "unit_of_measure" in df.columns:
        out["unit"] = df["unit_of_measure"]
    elif "unit_measure_1" in df.columns:
        out["unit"] = df["unit_measure_1"]

    if "measure_1" in df.columns:
        out["measure"] = df["measure_1"]
    elif "measure" in df.columns:
        out["measure"] = df["measure"]

    if "frequency_of_observation" in df.columns:
        out["frequency"] = df["frequency_of_observation"]
    elif "freq" in df.columns:
        out["frequency"] = df["freq"]

    out["date"] = pd.to_datetime(out["time_period"], errors="coerce")
    out["year"] = out["date"].dt.year
    out["month"] = out["date"].dt.month
    out["quarter"] = out["date"].dt.quarter

    out["indicator_name"] = "RFC_Index"
    out["dataset_id"] = DATASET
    out["source"] = "OECD API"

    return out


if __name__ == "__main__":
    print("Testing OECD GDP index API...\n")
    print(DATA_URL)
    print()

    rfc_raw_df = fetch_oecd_csv(DATA_URL, cache_file=CACHE_FILE, force_refresh=False)
    print("\nRaw shape:", rfc_raw_df.shape)
    print(rfc_raw_df.head())

    rfc_df = standardize_oecd_consumer_Price_Index(rfc_raw_df )
    print("\nClean shape:", rfc_df.shape)
    print(rfc_df.head())

    output_file = CACHE_DIR / f"RFC_Index_clean_{START_PERIOD}_{END_PERIOD}.csv"
    rfc_df.to_csv(output_file, index=False)
    print(f"\nSaved cleaned data to: {output_file}")

In [ ]:
rfc_df["date"] = pd.PeriodIndex(rfc_df["time_period"], freq="Q").to_timestamp()

In [ ]:
rfc_df = rfc_df.sort_values(["country", "date"])

rfc_df = (
    rfc_df.set_index("date")
          .groupby("country", group_keys=False)
          .resample("ME")
          .ffill()
          .reset_index()
)

In [ ]:
rfc_monthly_df = rfc_df.copy()

rfc_monthly_df["year"] = rfc_df["date"].dt.year
rfc_monthly_df["month"] = rfc_df["date"].dt.month
rfc_monthly_df["quarter"] = rfc_df["date"].dt.quarter

In [ ]:
rfc_monthly_df.head(5)

# EC Bulletin Fuel Prices Extraction

In [ ]:
import re
from io import BytesIO
from pathlib import Path


# =========================================================
# CONFIG
# =========================================================
FUEL_URL = (
    "https://energy.ec.europa.eu/document/download/"
    "906e60ca-8b6a-44e7-8589-652854d2fd3f_en"
    "?filename=Weekly_Oil_Bulletin_Prices_History_maticni_4web.xlsx"
)

OUTPUT_DIR = Path("fuel_data")
OUTPUT_DIR.mkdir(exist_ok=True)

RAW_FILE = OUTPUT_DIR / "weekly_oil_bulletin_prices_history.xlsx"
WEEKLY_OUTPUT = OUTPUT_DIR / "fuel_prices_weekly_long.csv"
MONTHLY_OUTPUT = OUTPUT_DIR / "fuel_prices_monthly.csv"

SHEET_NAME = "Prices with taxes"


# =========================================================
# STEP 1 - DOWNLOAD FILE
# =========================================================
def download_file(url: str, output_path: Path) -> None:
    response = requests.get(url, timeout=120)
    response.raise_for_status()

    with open(output_path, "wb") as f:
        f.write(response.content)

    print(f"Downloaded file to: {output_path}")


# =========================================================
# STEP 2 - LOAD AND CLEAN HEADER
# =========================================================
def load_fuel_sheet(file_path: Path, sheet_name: str) -> pd.DataFrame:
    """
    Loads the 'Prices with taxes' sheet.

    The workbook structure is:
    - row 1: machine-readable headers
    - row 2: verbose labels
    - row 3: units
    - row 4+: weekly data

    We keep row 1 as header and skip rows 2 and 3.
    """
    df = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        header=0,
        skiprows=[1, 2]
    )

    # Rename first column to date
    first_col = df.columns[0]
    df = df.rename(columns={first_col: "date"})

    # Remove fully empty columns
    df = df.dropna(axis=1, how="all")

    # Ensure date is datetime
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    # Remove rows without a valid date
    df = df[df["date"].notna()].copy()

    return df


# =========================================================
# STEP 3 - EXTRACT EURO-SUPER 95 + DIESEL INTO LONG FORMAT
# =========================================================
def extract_weekly_fuel_long(df: pd.DataFrame) -> pd.DataFrame:
    """
    Extracts only:
    - Euro-super 95
    - Diesel

    The sheet is wide, with one set of columns per country.
    We detect the relevant columns by name pattern and reshape to rows.
    """
    euro95_pattern = re.compile(r"^([A-Z0-9_]+)_price_with_tax_euro95$")
    diesel_pattern = re.compile(r"^([A-Z0-9_]+)_price_with_tax_diesel$")

    euro95_cols = {}
    diesel_cols = {}

    for col in df.columns:
        col_str = str(col)

        m1 = euro95_pattern.match(col_str)
        if m1:
            country_code = m1.group(1).replace("_", "")
            euro95_cols[country_code] = col_str
            continue

        m2 = diesel_pattern.match(col_str)
        if m2:
            country_code = m2.group(1).replace("_", "")
            diesel_cols[country_code] = col_str

    # Countries that have at least one of the two fuels
    all_country_codes = sorted(set(euro95_cols.keys()) | set(diesel_cols.keys()))

    records = []

    for country_code in all_country_codes:
        petrol_col = euro95_cols.get(country_code)
        diesel_col = diesel_cols.get(country_code)

        temp = pd.DataFrame({
            "date": df["date"],
            "country_code": country_code,
            "petrol_euro95": pd.to_numeric(df[petrol_col], errors="coerce") if petrol_col else pd.NA,
            "diesel": pd.to_numeric(df[diesel_col], errors="coerce") if diesel_col else pd.NA,
        })

        records.append(temp)

    weekly_long = pd.concat(records, ignore_index=True)

    # Drop rows where both prices are missing
    weekly_long = weekly_long.dropna(subset=["petrol_euro95", "diesel"], how="all").copy()

    # Add date helpers
    weekly_long["year"] = weekly_long["date"].dt.year
    weekly_long["month"] = weekly_long["date"].dt.month
    weekly_long["quarter"] = weekly_long["date"].dt.quarter

    return weekly_long


# =========================================================
# STEP 4 - AGGREGATE WEEKLY TO MONTHLY MEAN
# =========================================================
def aggregate_to_monthly_mean(weekly_long: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregates weekly prices to monthly averages per country.
    """
    monthly = (
        weekly_long
        .groupby(
            [
                "country_code",
                pd.Grouper(key="date", freq="ME")
            ],
            as_index=False
        )[["petrol_euro95", "diesel"]]
        .mean()
    )

    monthly["year"] = monthly["date"].dt.year
    monthly["month"] = monthly["date"].dt.month
    monthly["quarter"] = monthly["date"].dt.quarter

    monthly["source"] = "EU Commission Weekly Oil Bulletin"
    monthly["unit"] = "EUR per 1000 litres"

    return monthly


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    print("Downloading fuel price file...")
    download_file(FUEL_URL, RAW_FILE)

    print("Loading Excel sheet...")
    raw_fuel_df = load_fuel_sheet(RAW_FILE, SHEET_NAME)
    print("Raw shape:", raw_fuel_df.shape)
    print(raw_fuel_df.head(3))

    print("\nExtracting weekly Euro-super 95 and diesel...")
    fuel_weekly_long = extract_weekly_fuel_long(raw_fuel_df)
    print("Weekly long shape:", fuel_weekly_long.shape)
    print(fuel_weekly_long.head(10))

    print("\nAggregating weekly data to monthly mean...")
    fuel_monthly = aggregate_to_monthly_mean(fuel_weekly_long)
    print("Monthly shape:", fuel_monthly.shape)
    print(fuel_monthly.head(10))

    fuel_weekly_long.to_csv(WEEKLY_OUTPUT, index=False)
    fuel_monthly.to_csv(MONTHLY_OUTPUT, index=False)

    print(f"\nSaved weekly long file to: {WEEKLY_OUTPUT}")
    print(f"Saved monthly file to: {MONTHLY_OUTPUT}")

In [ ]:
START_DATE = "2022-01-01"
END_DATE = "2025-12-31"

fuel_monthly = fuel_monthly[
    (fuel_monthly["date"] >= START_DATE) &
    (fuel_monthly["date"] <= END_DATE)
]

In [ ]:
display(fuel_monthly)

In [ ]:
country_mapping = {
    "AT": "Austria",
    "BE": "Belgium",
    "BG": "Bulgaria",
    "CY": "Cyprus",
    "CZ": "Czech Republic",
    "DE": "Germany",
    "DK": "Denmark",
    "EE": "Estonia",
    "EU": "European Union",
    "EUR": "Euro Zone",
    "ES": "Spain",
    "FI": "Finland",
    "FR": "France",
    "GR": "Greece",
    "HR": "Croatia",
    "HU": "Hungary",
    "IE": "Ireland",
    "IT": "Italy",
    "LT": "Lithuania",
    "LU": "Luxembourg",
    "LV": "Latvia",
    "MT": "Malta",
    "NL": "Netherlands",
    "PL": "Poland",
    "PT": "Portugal",
    "RO": "Romania",
    "SE": "Sweden",
    "SI": "Slovenia",
    "SK": "Slovakia"
}

In [ ]:
fuel_monthly["country"] = fuel_monthly["country_code"].map(country_mapping)

In [ ]:
display(fuel_monthly["country"].unique())

In [ ]:
# --------------------------------------------------
# Calculate YoY growth for petrol_euro95 
# --------------------------------------------------

fuel_monthly = fuel_monthly.sort_values(["country", "date"])

fuel_monthly["petrol_euro95_yoy"] = (
    fuel_monthly
    .groupby("country")["petrol_euro95"]
    .pct_change(12) * 100
)

In [ ]:
%pip install pyarrow

# Saving All different Dataframes 

In [ ]:
# =========================================================
#  Saving all dataframes to desk 
# =========================================================


import os

# Create folder if it doesn't exist
os.makedirs("data", exist_ok=True)

# Save datasets
fuel_monthly.to_parquet("data/fuel_monthly.parquet", index=False)
Unemployment_df.to_parquet("data/unemployment.parquet", index=False)
CCI_df.to_parquet("data/cci.parquet", index=False)
CPI_df.to_parquet("data/cpi.parquet", index=False)
gdp_monthly_df.to_parquet("data/gdp_monthly.parquet", index=False)
rfc_monthly_df.to_parquet("data/rfc_monthly.parquet", index=False)
interest_rates_df.to_parquet("data/interest_rates.parquet", index=False)
